# Notebook 02b — SWaT & WADI Raw Data Cleaning (reconstructed, validated bit-exact)

This notebook was reconstructed from the original raw iTrust datasets after the notebook that
first produced `swat_normal.npy`, `swat_attack.npy`, `wadi_attack.npy` and
`wadi_attack_labels.npy` was lost. Every step below was reverse-engineered by inspecting the
already-cleaned arrays and re-deriving the exact transformation that produces them, then
**validated bit-exact** (max absolute difference `0.0`) against those known-good files.

It also produces `wadi_normal.npy`, which never existed before — WADI's within-plant MAML
evaluation (notebook 08) was previously unable to run its Isolation-Forest and few-shot support
sampling without it.

**Datasets required** (see `RAW_*` paths below — adjust to your local layout):
- SWaT.A1 & A2 (Dec 2015): `SWaT_Dataset_Normal_v1.xlsx`, `SWaT_Dataset_Attack_v0.xlsx`
- WADI.A2 (19 Nov 2019): `WADI_14days_new.csv`, `WADI_attackdataLABLE.csv`

Both are distributed by iTrust (SUTD) on request — see https://itrust.sutd.edu.sg/testbeds/.


In [1]:
import numpy as np
import pandas as pd
import openpyxl
import time
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

# Adjust these to wherever you've placed the raw iTrust files locally
RAW_SWAT_NORMAL = "data/raw/SWAT/SWaT_Dataset_Normal_v1.xlsx"
RAW_SWAT_ATTACK = "data/raw/SWAT/SWaT_Dataset_Attack_v0.xlsx"
RAW_WADI_NORMAL = "data/raw/WADI/WADI_14days_new.csv"
RAW_WADI_ATTACK = "data/raw/WADI/WADI_attackdataLABLE.csv"

WINDOW = 30
DOWNSAMPLE = 10


## 1 — SWaT: load the raw workbook

SWaT's raw files are Excel workbooks, not CSVs. Row 1 is a process-stage group header
(`P1`, `P2`, ...) that we skip; row 2 is the real column header
(`Timestamp`, 51 sensor/actuator tags, `Normal/Attack`). We read cell-by-cell in
read-only mode rather than via `pandas.read_excel`, since that turned out to be
noticeably faster on files this large (~500K rows).

In [2]:
def load_swat_workbook(path):
    t0 = time.time()
    wb = openpyxl.load_workbook(path, read_only=True)
    ws = wb[wb.sheetnames[0]]
    rows = ws.iter_rows(min_row=2, values_only=True)  # row2 = real header
    header = [str(h).strip() if h is not None else h for h in next(rows)]
    data = [r for r in rows]
    wb.close()
    print(f"  {path} -> {len(data)} rows, {len(header)} cols, {time.time()-t0:.0f}s")
    return header, data

def split_sensors_label(header, data):
    label_idx = header.index("Normal/Attack")
    sensor_idxs = [i for i in range(len(header)) if i not in (0, label_idx)]  # drop Timestamp + label
    n = len(data)
    X = np.empty((n, len(sensor_idxs)), dtype=np.float32)
    labels_raw = []
    for i, row in enumerate(data):
        for j, idx in enumerate(sensor_idxs):
            X[i, j] = row[idx]
        labels_raw.append(row[label_idx])
    return X, labels_raw

print("Loading SWaT Normal...")
h_n, d_n = load_swat_workbook(RAW_SWAT_NORMAL)
Xn_raw, labels_n = split_sensors_label(h_n, d_n)

print("Loading SWaT Attack...")
h_a, d_a = load_swat_workbook(RAW_SWAT_ATTACK)
Xa_raw, labels_a = split_sensors_label(h_a, d_a)

print(f"\nsensor matrix shapes: normal {Xn_raw.shape}, attack {Xa_raw.shape}")


Loading SWaT Normal...


/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


  data/raw/SWAT/SWaT_Dataset_Normal_v1.xlsx -> 495000 rows, 53 cols, 129s


Loading SWaT Attack...


  data/raw/SWAT/SWaT_Dataset_Attack_v0.xlsx -> 449919 rows, 53 cols, 125s



sensor matrix shapes: normal (495000, 51), attack (449919, 51)


## 2 — The `Normal/Attack` label bug

Scanning the real label column turns up **three** distinct string values, not two:

```
{'Normal', 'Attack', 'A ttack'}
```

`'A ttack'` has a genuine internal-space typo baked into the source file. Code that checks
`label == 'Attack'` silently drops every one of those rows as if they were normal — a real
undercount of attacks. The fix is to key off `label != 'Normal'` instead of `label == 'Attack'`,
which correctly captures both spellings.

In [3]:
labels_a_clean = [str(l).strip() for l in labels_a]
print("distinct raw labels:", set(labels_a_clean))

ya_bool = np.array([l != "Normal" for l in labels_a_clean])
print(f"attack rows (raw, pre-downsample): {ya_bool.sum()} / {len(ya_bool)}")


distinct raw labels: {'Attack', 'A ttack', 'Normal'}
attack rows (raw, pre-downsample): 54621 / 449919


## 3 — Downsample ×10

Two different rules, discovered by testing against the known-good downsampled outputs:

- **Sensor values**: simple stride-10 sampling (`x[::10]`), truncated to a multiple of 10 first
  so there's no partial trailing group (`x[:len(x)//10*10:10]`).
- **Labels**: **OR-aggregated** across each 10-row bucket — a downsampled timestep is flagged
  attack if *any* of its 10 raw rows were under attack. This is intentionally conservative:
  a brief attack that happens to fall on a row the stride sampler would have skipped is still
  caught.

In [4]:
def downsample_values(x):
    n = len(x)
    n_keep = n // DOWNSAMPLE
    return x[: n_keep * DOWNSAMPLE : DOWNSAMPLE]

def downsample_labels_or(y):
    n = len(y)
    n_keep = n // DOWNSAMPLE
    return y[: n_keep * DOWNSAMPLE].reshape(n_keep, DOWNSAMPLE).any(axis=1)

Xn_ds = downsample_values(Xn_raw)
Xa_ds = downsample_values(Xa_raw)
ya_ds = downsample_labels_or(ya_bool)
print("downsampled shapes:", Xn_ds.shape, Xa_ds.shape, ya_ds.shape)


downsampled shapes: (49500, 51) (44991, 51) (44991,)


## 4 — MinMax scaling, fit on normal data only

Standard rule to avoid leakage: the scaler learns its min/max purely from the attack-free
normal trace, then transforms both normal and attack data with those same bounds. Attack
values that exceed the training range get clipped to `[0, 1]`.

In [5]:
scaler_swat = MinMaxScaler()
scaler_swat.fit(Xn_ds)
swat_normal = scaler_swat.transform(Xn_ds).astype(np.float32)
swat_attack = np.clip(scaler_swat.transform(Xa_ds), 0, 1).astype(np.float32)
swat_attack_labels = ya_ds.astype(np.int8)

print("swat_normal:", swat_normal.shape, " swat_attack:", swat_attack.shape,
      " labels:", swat_attack_labels.shape, " attacks:", swat_attack_labels.sum())


swat_normal: (49500, 51)  swat_attack: (44991, 51)  labels: (44991,)  attacks: 5493


## 5 — Validate against the known-good arrays

If this reconstruction is faithful, it should reproduce `swat_normal.npy`, `swat_attack.npy`
and `swat_attack_labels.npy` **exactly** — not approximately.

In [6]:
gt_normal = np.load("swat_normal.npy")
gt_attack = np.load("swat_attack.npy")
gt_labels = np.load("swat_attack_labels.npy")

print("normal  max abs diff:", np.abs(swat_normal - gt_normal).max(),
      " exact match:", np.array_equal(swat_normal, gt_normal))
print("attack  max abs diff:", np.abs(swat_attack - gt_attack).max(),
      " exact match:", np.array_equal(swat_attack, gt_attack))
print("labels  exact match:", np.array_equal(swat_attack_labels, gt_labels))

assert np.array_equal(swat_normal, gt_normal)
assert np.array_equal(swat_attack, gt_attack)
assert np.array_equal(swat_attack_labels, gt_labels)
print("\n*** SWaT reconstruction verified bit-exact ***")


normal  max abs diff: 0.0  exact match: True


attack  max abs diff: 0.0  exact match: True
labels  exact match: True

*** SWaT reconstruction verified bit-exact ***


## 6 — WADI: load the raw CSVs

WADI's two files have inconsistent formatting even between themselves: the normal file's
header is a clean first row, but the attack file has a spurious numeric index row *before*
its real header (`header=1` skips it), and its `Row`/`Date` column names carry trailing
spaces.

In [7]:
t0 = time.time()
wadi_normal_df = pd.read_csv(RAW_WADI_NORMAL, low_memory=False)
print("WADI normal:", wadi_normal_df.shape, f"{time.time()-t0:.0f}s")

t0 = time.time()
wadi_attack_df = pd.read_csv(RAW_WADI_ATTACK, header=1, low_memory=False)
wadi_attack_df.columns = [c.strip() for c in wadi_attack_df.columns]
print("WADI attack:", wadi_attack_df.shape, f"{time.time()-t0:.0f}s")


WADI normal: (784571, 130) 16s


WADI attack: (172803, 131) 4s


## 7 — WADI cleaning: drop the 4 empty channels, interpolate gaps, remap the label polarity

Scanning all 127 raw sensor columns (excluding `Row`/`Date`/`Time`) for fully-`NaN` columns
finds exactly four — the "four empty channels":

```
2_LS_001_AL, 2_LS_002_AL, 2_P_001_STATUS, 2_P_002_STATUS
```

127 − 4 = 123, matching the known feature count. A further four columns are only *partially*
gapped and get linearly interpolated rather than dropped:

```
1_AIT_002_PV, 1_AIT_004_PV, 2B_AIT_004_PV, 3_AIT_004_PV
```

The attack file's own label column is literally named
`Attack LABLE (1:No Attack, -1:Attack)` — the "−1 polarity" is this reversed convention
(1 = normal, −1 = attack), remapped here to the standard boolean where `True` = attack. The
attack file also has two fully-blank trailing rows that get dropped before anything else.

In [8]:
EMPTY_COLS = ['2_LS_001_AL', '2_LS_002_AL', '2_P_001_STATUS', '2_P_002_STATUS']
INTERP_COLS = ['1_AIT_002_PV', '1_AIT_004_PV', '2B_AIT_004_PV', '3_AIT_004_PV']

def clean_wadi_sensors(df):
    df = df.drop(columns=['Row', 'Date', 'Time'])
    df = df.drop(columns=EMPTY_COLS)
    for c in INTERP_COLS:
        df[c] = df[c].interpolate(method='linear', limit_direction='both')
    assert df.isna().sum().sum() == 0
    return df

wadi_normal_clean = clean_wadi_sensors(wadi_normal_df)
sensor_cols = wadi_normal_clean.columns.tolist()
print("WADI normal sensors:", wadi_normal_clean.shape)

wadi_attack_df = wadi_attack_df.dropna(subset=['Row'])  # drop the trailing blank rows
label_col = [c for c in wadi_attack_df.columns if 'Attack LABLE' in c][0]
wadi_attack_is_attack = (wadi_attack_df[label_col].to_numpy() == -1)
print(f"attack rows (raw, pre-downsample): {wadi_attack_is_attack.sum()} / {len(wadi_attack_is_attack)}")

wadi_attack_clean = clean_wadi_sensors(wadi_attack_df.drop(columns=[label_col]))
assert wadi_attack_clean.columns.tolist() == sensor_cols
print("WADI attack sensors:", wadi_attack_clean.shape)


WADI normal sensors: (784571, 123)
attack rows (raw, pre-downsample): 9977 / 172801


WADI attack sensors: (172801, 123)


## 8 — Downsample ×10 and scale

Same two-rule pattern as SWaT: sensor values by plain stride-10, labels by OR-aggregation
across each 10-row bucket. Scaling again fits MinMax on normal data only.

In [9]:
Xn_wadi = downsample_values(wadi_normal_clean.to_numpy(dtype=np.float32))
Xa_wadi = downsample_values(wadi_attack_clean.to_numpy(dtype=np.float32))
ya_wadi = downsample_labels_or(wadi_attack_is_attack)

scaler_wadi = MinMaxScaler()
scaler_wadi.fit(Xn_wadi)
wadi_normal = scaler_wadi.transform(Xn_wadi).astype(np.float32)
wadi_attack = np.clip(scaler_wadi.transform(Xa_wadi), 0, 1).astype(np.float32)
wadi_attack_labels = ya_wadi.astype(np.int8)

print("wadi_normal:", wadi_normal.shape, " wadi_attack:", wadi_attack.shape,
      " labels:", wadi_attack_labels.shape, " attacks:", wadi_attack_labels.sum())


wadi_normal: (78457, 123)  wadi_attack: (17280, 123)  labels: (17280,)  attacks: 1010


## 9 — Validate against the known-good arrays

`wadi_normal.npy` didn't exist before this notebook, so there's nothing to check it against —
but `wadi_attack.npy` and `wadi_attack_labels.npy` did, and should match exactly.

In [10]:
gt_wadi_attack = np.load("wadi_attack.npy")
gt_wadi_labels = np.load("wadi_attack_labels.npy")

print("attack  max abs diff:", np.abs(wadi_attack - gt_wadi_attack).max(),
      " exact match:", np.array_equal(wadi_attack, gt_wadi_attack))
print("labels  exact match:", np.array_equal(wadi_attack_labels, gt_wadi_labels))

assert np.array_equal(wadi_attack, gt_wadi_attack)
assert np.array_equal(wadi_attack_labels, gt_wadi_labels)
print("\n*** WADI reconstruction verified bit-exact ***")


attack  max abs diff: 0.0  exact match: True
labels  exact match: True

*** WADI reconstruction verified bit-exact ***


## 10 — Save

`swat_normal.npy` / `swat_attack.npy` / `swat_attack_labels.npy` already exist and are
confirmed identical above, so they're not overwritten here. `wadi_normal.npy` is new.

In [11]:
np.save("wadi_normal.npy", wadi_normal)
print("saved wadi_normal.npy", wadi_normal.shape)


saved wadi_normal.npy (78457, 123)
